# 04 — Merge MosquitoFusion + VisText-Mosquito -> Final Training Set (Multi-Class)
Combines both real datasets (no drone data) into **one folder that keeps every original class**
(Breeding Place, Mosquito, Mosquito Swarm, Bottle, Coconut-Exocarp, Drain-Inlet, Tire, Vase) under
one unified numbering, instead of filtering down to breeding-site images only.

This matters for Notebook 05: a binary "breeding_site vs no_breeding_site" classifier needs real
negative examples (mosquito/swarm-only photos) sitting alongside the positives, not a folder that
only ever contains positives. This version writes a `data.yaml` so Notebook 05 can read the real
class list directly instead of guessing.

In [36]:
import os, shutil, random
from collections import Counter
import pandas as pd
import cv2

import os, yaml

def read_data_yaml(root):
    """Reads a Roboflow-style data.yaml and returns the class name list."""
    yaml_path = os.path.join(root, "data.yaml")
    if not os.path.isfile(yaml_path):
        print(f"[!] No data.yaml found in {root}")
        return None
    with open(yaml_path) as f:
        data = yaml.safe_load(f)
    return data.get("names")

def detect_splits(root):
    """Finds split folders (train/valid/test, or train/val/test, etc.) that contain images+labels."""
    found = {}
    for name in os.listdir(root):
        split_dir = os.path.join(root, name)
        if not os.path.isdir(split_dir):
            continue
        img_dir = os.path.join(split_dir, "images")
        lbl_dir = os.path.join(split_dir, "labels")
        if os.path.isdir(img_dir) and os.path.isdir(lbl_dir):
            found[name] = {"images": img_dir, "labels": lbl_dir}
    return found

PATH_MOSQUITOFUSION = "data/raw/MosquitoFusion Dataset"
PATH_VISTEXT_DETECT  = os.path.join("data/raw/VisText-Mosquito A Multimodal Dataset for Mosquito", "Breeding Place Detection")
FINAL_ROOT           = "data/processed/cnn_breeding_site"

for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(FINAL_ROOT, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(FINAL_ROOT, split, "labels"), exist_ok=True)

mf_classes = read_data_yaml(PATH_MOSQUITOFUSION)
vt_classes = read_data_yaml(PATH_VISTEXT_DETECT)
print("MosquitoFusion classes:", mf_classes)
print("VisText-Mosquito classes:", vt_classes)

MosquitoFusion classes: ['Breeding Place', 'Mosquito', 'Mosquito Swarm']
VisText-Mosquito classes: ['Bottle', 'Coconut-Exocarp', 'Drain-Inlet', 'Tire', 'Vase']


## 1. Define the unified 8-class scheme
Every class from both datasets is kept, just renumbered onto one shared list so there's no id
clash between the two source datasets.

In [37]:
UNIFIED_CLASSES = ["Breeding Place", "Mosquito", "Mosquito Swarm",
                    "Bottle", "Coconut-Exocarp", "Drain-Inlet", "Tire", "Vase"]
unified_index = {name.lower(): i for i, name in enumerate(UNIFIED_CLASSES)}

def build_id_map(dataset_classes):
    """Maps each of a dataset's own class ids to the unified class id, by matching name."""
    id_map = {}
    for local_id, name in enumerate(dataset_classes or []):
        key = name.strip().lower()
        if key in unified_index:
            id_map[local_id] = unified_index[key]
        else:
            print(f"[!] Class '{name}' not found in UNIFIED_CLASSES - its objects will be dropped. "
                  f"Add it to UNIFIED_CLASSES above if it should be kept.")
    return id_map

mf_id_map = build_id_map(mf_classes)
vt_id_map = build_id_map(vt_classes)
print("MosquitoFusion id map (local -> unified):", mf_id_map)
print("VisText-Mosquito id map (local -> unified):", vt_id_map)

MosquitoFusion id map (local -> unified): {0: 0, 1: 1, 2: 2}
VisText-Mosquito id map (local -> unified): {0: 3, 1: 4, 2: 5, 3: 6, 4: 7}


## 2. Copy ALL images (not filtered) + remap every label line to the unified id
Unlike the previous version, this keeps mosquito/swarm-only images too - they become your
negative class in Notebook 05.

In [38]:
def copy_and_remap_all(src_root, id_map, out_stem_prefix):
    src_splits = detect_splits(src_root)
    totals = {}
    for split, dirs in src_splits.items():
        norm_split = {"val": "valid"}.get(split, split)
        if norm_split not in ("train", "valid", "test"):
            print(f"[!] Unrecognised split name '{split}' in {src_root}, skipping")
            continue
        dst_img = os.path.join(FINAL_ROOT, norm_split, "images")
        dst_lbl = os.path.join(FINAL_ROOT, norm_split, "labels")
        kept = 0
        for lbl_file in os.listdir(dirs["labels"]):
            lines = open(os.path.join(dirs["labels"], lbl_file)).read().splitlines()
            keep_lines = []
            for line in lines:
                if not line.strip():
                    continue
                parts = line.split()
                local_id = int(parts[0])
                if local_id in id_map:
                    keep_lines.append(" ".join([str(id_map[local_id])] + parts[1:]))
            # Copy the image REGARDLESS of whether it had any keepable objects -
            # an empty/negative label file is still a valid (negative) training example.
            stem = f"{out_stem_prefix}_{lbl_file.rsplit('.',1)[0]}"
            for ext in [".jpg", ".jpeg", ".png"]:
                src_img_path = os.path.join(dirs["images"], lbl_file.rsplit(".",1)[0] + ext)
                if os.path.isfile(src_img_path):
                    shutil.copy(src_img_path, os.path.join(dst_img, stem + ext))
                    break
            with open(os.path.join(dst_lbl, stem + ".txt"), "w") as f:
                f.write("\n".join(keep_lines))
            kept += 1
        totals[norm_split] = kept
        print(f"{src_root} [{split} -> {norm_split}]: {kept} images merged (all classes kept)")
    return totals

mf_totals = copy_and_remap_all(PATH_MOSQUITOFUSION, mf_id_map, "mf")
vt_totals = copy_and_remap_all(PATH_VISTEXT_DETECT, vt_id_map, "vt")

# Write data.yaml so Notebook 05 (and anything else) reads the real class list, no fallback guessing
import yaml
with open(os.path.join(FINAL_ROOT, "data.yaml"), "w") as f:
    yaml.safe_dump({"names": UNIFIED_CLASSES, "nc": len(UNIFIED_CLASSES)}, f)
print("\nWrote", os.path.join(FINAL_ROOT, "data.yaml"), "with classes:", UNIFIED_CLASSES)

data/raw/MosquitoFusion Dataset [test -> test]: 51 images merged (all classes kept)
data/raw/MosquitoFusion Dataset [train -> train]: 1053 images merged (all classes kept)
data/raw/MosquitoFusion Dataset [valid -> valid]: 100 images merged (all classes kept)
data/raw/VisText-Mosquito A Multimodal Dataset for Mosquito\Breeding Place Detection [test -> test]: 183 images merged (all classes kept)
data/raw/VisText-Mosquito A Multimodal Dataset for Mosquito\Breeding Place Detection [train -> train]: 3871 images merged (all classes kept)
data/raw/VisText-Mosquito A Multimodal Dataset for Mosquito\Breeding Place Detection [valid -> valid]: 371 images merged (all classes kept)

Wrote data/processed/cnn_breeding_site\data.yaml with classes: ['Breeding Place', 'Mosquito', 'Mosquito Swarm', 'Bottle', 'Coconut-Exocarp', 'Drain-Inlet', 'Tire', 'Vase']


In [39]:
# --- NEW: Inject Intel Background Images (Negative Class) ---
import glob, shutil
from sklearn.model_selection import train_test_split

INTEL_DIR = "data/raw/Intel Image Classification/seg_train/seg_train"
if not os.path.exists(INTEL_DIR):
    INTEL_DIR = "data/raw/Intel Image Classification/seg_train"

intel_images = []
for cat in ["buildings", "forest", "street", "sea", "glacier", "mountain"]:
    cat_dir = os.path.join(INTEL_DIR, cat)
    if os.path.exists(cat_dir):
        intel_images.extend(glob.glob(os.path.join(cat_dir, "*.jpg")))

# 80/10/10 Split for Backgrounds
itrain, itemp = train_test_split(intel_images, test_size=0.2, random_state=42)
ival, itest = train_test_split(itemp, test_size=0.5, random_state=42)

intel_totals = {}
for split_name, imgs in zip(["train", "valid", "test"], [itrain, ival, itest]):
    dst_img = os.path.join(FINAL_ROOT, split_name, "images")
    dst_lbl = os.path.join(FINAL_ROOT, split_name, "labels")
    for i, img_path in enumerate(imgs):
        stem = f"intel_{split_name}_{i}"
        shutil.copy(img_path, os.path.join(dst_img, stem + ".jpg"))
        # Empty text file means 'no objects' (safe/background) in YOLO format
        with open(os.path.join(dst_lbl, stem + ".txt"), "w") as f:
            pass
    intel_totals[split_name] = len(imgs)
    print(f"Intel Backgrounds -> {split_name}: {len(imgs)} negative images added")

Intel Backgrounds -> train: 5475 negative images added
Intel Backgrounds -> valid: 684 negative images added
Intel Backgrounds -> test: 685 negative images added


## 3. Combined real (original) image count — before augmentation

In [40]:
combined = pd.DataFrame({"MosquitoFusion": pd.Series(mf_totals),"VisText-Mosquito": pd.Series(vt_totals),"Intel-Background": pd.Series(intel_totals)}).fillna(0).astype(int)
combined["Total"] = combined.sum(axis=1)
combined.loc["ALL SPLITS"] = combined.sum()
combined

,MosquitoFusion,VisText-Mosquito,Intel-Background,Total
test,51,183,685,919
train,1053,3871,5475,10399
valid,100,371,684,1155
ALL SPLITS,1204,4425,6844,12473


## 4. Recalculate the augmentation multiplier needed for ~4,000 training samples

In [41]:
TARGET_IMAGES = 4500
combined_train_original = int(combined.loc['train', 'MosquitoFusion']) + int(combined.loc['train', 'VisText-Mosquito']) if 'train' in combined.index else 0

if combined_train_original > 0:
    multiplier_needed = TARGET_IMAGES / combined_train_original
    print(f'Combined ORIGINAL train images (MosquitoFusion + VisText-Mosquito): {combined_train_original}')
    print(f'Target training samples: {TARGET_IMAGES}')
    print(f'Augmentation multiplier needed: {multiplier_needed:.2f}x')
else:
    print('No merged train images found yet - check Section 2 output above for errors.')


Combined ORIGINAL train images (MosquitoFusion + VisText-Mosquito): 4924
Target training samples: 4500
Augmentation multiplier needed: 0.91x


## 5. Run augmentation on the merged train split (uses `augment_one_image` from Notebook 02)

In [42]:
import albumentations as A

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.6, border_mode=cv2.BORDER_REFLECT101),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.GaussNoise(p=0.2),
    A.ToGray(p=0.15),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def augment_one_image(img_path, label_path, n_copies, out_img_dir, out_lbl_dir, stem):
    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    bboxes, class_labels = [], []
    for line in open(label_path):
        parts = line.split()
        if len(parts) == 5:
            cid, x, y, w, h = parts
            bboxes.append([float(x), float(y), float(w), float(h)])
            class_labels.append(int(cid))
    for i in range(n_copies):
        out = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        aug_img = cv2.cvtColor(out['image'], cv2.COLOR_RGB2BGR)
        out_stem = f"{stem}_aug{i}"
        cv2.imwrite(os.path.join(out_img_dir, out_stem + ".jpg"), aug_img)
        with open(os.path.join(out_lbl_dir, out_stem + ".txt"), "w") as f:
            for cid, box in zip(out['class_labels'], out['bboxes']):
                f.write(f"{cid} {' '.join(f'{v:.6f}' for v in box)}\n")

n_copies = round(multiplier_needed) if combined_train_original else 3
train_img_dir = os.path.join(FINAL_ROOT, "train", "images")
train_lbl_dir = os.path.join(FINAL_ROOT, "train", "labels")
generated = 0
for lbl_file in list(os.listdir(train_lbl_dir)):
    if lbl_file.startswith('intel_'):
        continue
    stem = lbl_file.rsplit(".", 1)[0]
    for ext in [".jpg", ".jpeg", ".png"]:
        img_path = os.path.join(train_img_dir, stem + ext)
        if os.path.isfile(img_path):
            augment_one_image(img_path, os.path.join(train_lbl_dir, lbl_file),
                               n_copies, train_img_dir, train_lbl_dir, stem)
            generated += n_copies
            break

print(f"Augmented copies generated per original image: {n_copies}")
print(f"Total NEW augmented images added: {generated}")
print(f"Final train set size (originals + augmented): {combined_train_original + generated}")

Augmented copies generated per original image: 1
Total NEW augmented images added: 4924
Final train set size (originals + augmented): 9848


## 6. Write the final dataset card

In [44]:
card = f'''# DengueSense LK — CNN Training Dataset Card (v4, multi-class merge)

## Sources (real, original images)
- MosquitoFusion Dataset (Kaggle: faiyazabdullah/mosquitofusion-dataset) — ALL 3 classes kept
  (Breeding Place, Mosquito, Mosquito Swarm).
- VisText-Mosquito, Breeding Place Detection subset (Sayeedi et al., 2025) — ALL 5 container
  classes kept (Bottle, Coconut-Exocarp, Drain-Inlet, Tire, Vase).
- Drone CSV dataset (Tanzania): reviewed, NOT used (per project decision).
- VisText-Mosquito Water Surface Segmentation subset: NOT used (different task, out of current scope).

## Unified class scheme (see data.yaml in this folder)
0 = Breeding Place, 1 = Mosquito, 2 = Mosquito Swarm,
3 = Bottle, 4 = Coconut-Exocarp, 5 = Drain-Inlet, 6 = Tire, 7 = Vase

Unlike earlier versions, this export keeps EVERY image from both datasets (not filtered to
breeding-site only), so Mosquito/Mosquito Swarm-only images act as real negative examples for
downstream binary classification (Notebook 05 groups classes 0,3,4,5,6,7 as "breeding_site"
and 1,2 as "no_breeding_site").

## Combined original image counts (all classes)
{combined.to_string()}

## Augmentation
- Applied to the TRAIN split only, using flip/rotation/brightness/crop/noise/grayscale.
- Bounding boxes remapped, not discarded, during augmentation - multi-object images keep all objects.
- Validation and test sets contain ONLY original, non-augmented images.
'''

with open(os.path.join(FINAL_ROOT, "dataset_card.md"), "w") as f:
    f.write(card)
print(card)

# DengueSense LK — CNN Training Dataset Card (v4, multi-class merge)

## Sources (real, original images)
- MosquitoFusion Dataset (Kaggle: faiyazabdullah/mosquitofusion-dataset) — ALL 3 classes kept
  (Breeding Place, Mosquito, Mosquito Swarm).
- VisText-Mosquito, Breeding Place Detection subset (Sayeedi et al., 2025) — ALL 5 container
  classes kept (Bottle, Coconut-Exocarp, Drain-Inlet, Tire, Vase).
- Drone CSV dataset (Tanzania): reviewed, NOT used (per project decision).
- VisText-Mosquito Water Surface Segmentation subset: NOT used (different task, out of current scope).

## Unified class scheme (see data.yaml in this folder)
0 = Breeding Place, 1 = Mosquito, 2 = Mosquito Swarm,
3 = Bottle, 4 = Coconut-Exocarp, 5 = Drain-Inlet, 6 = Tire, 7 = Vase

Unlike earlier versions, this export keeps EVERY image from both datasets (not filtered to
breeding-site only), so Mosquito/Mosquito Swarm-only images act as real negative examples for
downstream binary classification (Notebook 05 groups

Your MobileNetV3 CNN training script should now read directly from `data/processed/cnn_breeding_site/`.